### Equipo
* Rocío Sánchez Solórzano 	2043109	IB
* Alisson Michelle Sepúlveda Torres	2115253 IB
* Benito Woo Zozaya	2177866	IB
* David Palma Cabañez	2177805	IB


In [1]:
#Librerias
import kagglehub
import numpy as np
import kagglehub
import cv2
import os
%pip install pydicom
import pydicom
from google.colab.patches import cv_imshow
from matplotlib import pyplot as plt
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.5 MB/s eta 0:00:00


In [ ]:
#1 Cargar imagenes
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print("Path to dataset files:", image_path)
dicom_data = pydicom.dcmread(image_path)
#normalizamos la imagen para pasarla de 16 bits a 8 bits
dicom_image = cv2.normalize(dicom_data.pixel_array, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
dicom_image_high = cv2.normalize(dicom_data.pixel_array, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_32F)

In [ ]:
#Filtros paso bajo (media, gaussiano, mediana), usando diferentes tamaños de kernel (3x3, 5x5, 11x11).
kernel = 3
media_filtered = cv2.blur(dicom_image, (kernel, kernel))
gaussian_filtered = cv2.GaussianBlur(dicom_image_high, (kernel, kernel), 0)
median_filtered = cv2.medianBlur(dicom_image, kernel)

plt.figure(figsize=(11, 6))
plt.subplot(2, 2, 1)
plt.imshow(dicom_image, cmap='gray')
plt.title('Original Image')
plt.subplot(2, 2, 2)
plt.imshow(media_filtered, cmap='gray')
plt.title(f'Filtro Media ({kernel}x{kernel})')
plt.subplot(2, 2, 3)
plt.imshow(gaussian_filtered, cmap='gray')
plt.title(f'Filtro Gaussiano ({kernel}x{kernel})')
plt.subplot(2, 2, 4)
plt.imshow(median_filtered, cmap='gray')
plt.title(f'Filtro Mediana ({kernel}x{kernel})')

plt.tight_layout()
plt.show()


In [ ]:
#Detección de Bordes

img_laplace = cv2.Laplacian(dicom_image_high, cv2.CV_32F, ksize=3)

sobelx = cv2.Sobel(dicom_image_high, cv2.CV_32F, 1, 0, ksize=3)
sobely = cv2.Sobel(dicom_image_high, cv2.CV_32F, 0, 1, ksize=3)
abs_sobelx = np.absolute(sobelx)
abs_sobely = np.absolute(sobely)
img_sobel = cv2.addWeighted(abs_sobelx, 0.5, abs_sobely, 0.5, 0)

kernels_kirsch = [
    np.array([[ 5,  5,  5], [-3,  0, -3], [-3, -3, -3]]), # N
    np.array([[-3,  5,  5], [-3,  0,  5], [-3, -3, -3]]), # NE
    np.array([[-3, -3,  5], [-3,  0,  5], [-3, -3,  5]]), # E
    np.array([[-3, -3, -3], [-3,  0,  5], [-3,  5,  5]]), # SE
    np.array([[-3, -3, -3], [-3,  0, -3], [ 5,  5,  5]]), # S
    np.array([[-3, -3, -3], [ 5,  0, -3], [ 5,  5, -3]]), # SW
    np.array([[ 5, -3, -3], [ 5,  0, -3], [ 5, -3, -3]]), # W
    np.array([[ 5,  5, -3], [ 5,  0, -3], [-3, -3, -3]])  # NW
]

# Inicializamos la matriz de acumulación en float32 del mismo tamaño que la imagen
img_kirsch = np.zeros_like(dicom_image_high)

# Aplicamos cada uno de los 8 filtros y guardamos el gradiente máximo por píxel
for k in kernels_kirsch:
    # Usamos cv2.CV_32F para no perder precisión durante las convoluciones
    resultado = cv2.filter2D(dicom_image_high, cv2.CV_32F, k)
    img_kirsch = np.maximum(img_kirsch, resultado)

def normalizar_por_percentil(imagen_borde, percentil=99.5): #se utiliza para evitar el contraste de la esquina
    limite_superior = np.percentile(imagen_borde, percentil)
    imagen_recortada = np.clip(imagen_borde, 0, limite_superior)
    return cv2.normalize(imagen_recortada, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)


laplace_vis = normalizar_por_percentil(np.absolute(img_laplace), percentil=99.5)
sobel_vis = normalizar_por_percentil(img_sobel, percentil=99.5)
kirsch_vis = normalizar_por_percentil(img_kirsch, percentil=99.5)

plt.figure(figsize=(15, 5))


plt.subplot(2, 2, 1)
plt.imshow(dicom_image, cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.subplot(2, 2, 2)
plt.imshow(laplace_vis, cmap='gray')
plt.title(f'Bordes Laplace')
plt.axis('off')
plt.subplot(2, 2, 3)
plt.imshow(sobel_vis, cmap='gray')
plt.title(f'Bordes Sobel')
plt.axis('off')
plt.subplot(2, 2, 4)
plt.imshow(kirsch_vis, cmap='gray')
plt.title(f'Filtro Kirsh')
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#Mascara personalizada, utilizaremos el filtro LoG, Laplacian of Gaussian para esto
kernel_gauss = 7 #cuanto más alto el tamaño del kernel más ruido limpia
kernel_laplace = 3
gaussian_filtered = cv2.GaussianBlur(dicom_image_high, (kernel_gauss, kernel_gauss), 0)
img_laplace = cv2.Laplacian(gaussian_filtered, cv2.CV_32F, kernel_laplace)
img_LoG = normalizar_por_percentil(np.absolute(img_laplace), percentil=99.5)

threshold =35


_, binary_segmentation = cv2.threshold(img_LoG, threshold, 255, cv2.THRESH_BINARY)
img= cv2.bitwise_and(dicom_image, dicom_image, mask=binary_segmentation)

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.imshow(dicom_image, cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(img, cmap='gray')
plt.title(f'Laplacian of Gaussian')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Filtros en el Dominio de la Frecuencia con Parámetros Modificables
def fourier_pasa_bajas(img_gris, radio):
    # Transformada y centrado
    transformada = np.fft.fft2(img_gris)
    shift = np.fft.fftshift(transformada)

    rows, cols = img_gris.shape
    crow, ccol = rows // 2, cols // 2

    # Crear Máscara Pasa Bajas (Círculo)
    mask = np.zeros((rows, cols), np.float32)
    y, x = np.ogrid[-crow:rows-crow, -ccol:cols-ccol]
    distancia = np.sqrt(x**2 + y**2)
    mask[distancia <= radio] = 1

    # Filtrar y aplicar transformada inversa
    filtrado = shift * mask
    ishift = np.fft.ifftshift(filtrado)
    img_back = np.abs(np.fft.ifft2(ishift))

    return mask, img_back

def fourier_pasa_altas_anillo(img_gris, radio_interno, radio_externo):
    transformada = np.fft.fft2(img_gris)
    shift = np.fft.fftshift(transformada)

    rows, cols = img_gris.shape
    crow, ccol = rows // 2, cols // 2

    # Crear Máscara de Anillo (Pasa frecuencias medias-altas)
    mask = np.zeros((rows, cols), np.float32)
    y, x = np.ogrid[-crow:rows-crow, -ccol:cols-ccol]
    dist_cuadrado = x**2 + y**2
    # El anillo activa los píxeles entre el radio interno y externo
    mask[(dist_cuadrado >= radio_interno**2) & (dist_cuadrado <= radio_externo**2)] = 1

    filtrado = shift * mask
    ishift = np.fft.ifftshift(filtrado)
    img_back = np.abs(np.fft.ifft2(ishift))

    return mask, img_back

def fourier_pasa_altas_cruz(img_gris, radio_centro_excluido, grosor):
    transformada = np.fft.fft2(img_gris)
    shift = np.fft.fftshift(transformada)

    rows, cols = img_gris.shape
    crow, ccol = rows // 2, cols // 2

    # Crear Máscara de Cruz (Pasa frecuencias de bordes verticales y horizontales)
    mask = np.zeros((rows, cols), np.float32)

    # Activar franjas vertical y horizontal de la cruz
    mask[crow - grosor : crow + grosor, :] = 1
    mask[:, ccol - grosor : ccol + grosor] = 1

    # Excluir las frecuencias muy bajas del centro (dibujando un círculo negro en medio)
    y, x = np.ogrid[-crow:rows-crow, -ccol:cols-ccol]
    distancia = np.sqrt(x**2 + y**2)
    mask[distancia <= radio_centro_excluido] = 0

    filtrado = shift * mask
    ishift = np.fft.ifftshift(filtrado)
    img_back = np.abs(np.fft.ifft2(ishift))

    return mask, img_back

# ==========================================
# 2. Función para Graficar las Máscaras y Resultados
# ==========================================

def mostrar_galeria_fourier(imagen, titulo_imagen):
    # Generar los 4 filtros requeridos
    mask_pb30, res_pb30 = fourier_pasa_bajas(imagen, radio=30)
    mask_pb80, res_pb80 = fourier_pasa_bajas(imagen, radio=80)
    mask_pa_anillo, res_pa_anillo = fourier_pasa_altas_anillo(imagen, radio_interno=15, radio_externo=120)
    mask_pa_cruz, res_pa_cruz = fourier_pasa_altas_cruz(imagen, radio_centro_excluido=20, grosor=6)

    # Normalizar resultados para visualización homogénea
    res_pb30_norm = cv2.normalize(res_pb30, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    res_pb80_norm = cv2.normalize(res_pb80, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    res_pa_anillo_norm = cv2.normalize(res_pa_anillo, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    res_pa_cruz_norm = cv2.normalize(res_pa_cruz, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

    # Configuración de la cuadrícula de visualización
    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'Procesamiento en Frecuencia - {titulo_imagen}', fontsize=18, fontweight='bold')

    # Fila 1: Las Máscaras Espectrales Aplicadas
    axs[0, 0].imshow(mask_pb30, cmap='gray')
    axs[0, 0].set_title('Máscara Pasa Bajas (R=30)')
    axs[0, 0].axis('off')

    axs[0, 1].imshow(mask_pb80, cmap='gray')
    axs[0, 1].set_title('Máscara Pasa Bajas (R=80)')
    axs[0, 1].axis('off')

    axs[0, 2].imshow(mask_pa_anillo, cmap='gray')
    axs[0, 2].set_title('Máscara Pasa Altas Anillo')
    axs[0, 2].axis('off')

    axs[0, 3].imshow(mask_pa_cruz, cmap='gray')
    axs[0, 3].set_title('Máscara Pasa Altas Cruz')
    axs[0, 3].axis('off')

    # Fila 2: Las Imágenes Transformadas Inversamente (Resultado)
    axs[1, 0].imshow(res_pb30_norm, cmap='gray')
    axs[1, 0].set_title('Resultado Pasa Bajas (R=30)')
    axs[1, 0].axis('off')

    axs[1, 1].imshow(res_pb80_norm, cmap='gray')
    axs[1, 1].set_title('Resultado Pasa Bajas (R=80)')
    axs[1, 1].axis('off')

    axs[1, 2].imshow(res_pa_anillo_norm, cmap='gray')
    axs[1, 2].set_title('Resultado Pasa Altas Anillo')
    axs[1, 2].axis('off')

    axs[1, 3].imshow(res_pa_cruz_norm, cmap='gray')
    axs[1, 3].set_title('Resultado Pasa Altas Cruz')
    axs[1, 3].axis('off')

    plt.tight_layout()
    plt.show()





In [ ]:
# Procesamos directamente usando la imagen de 8 bits normalizada de tu radiografía
mostrar_galeria_fourier(dicom_image, "Imagen 1: Radiografía de Tórax DICOM")

In [ ]:
# Subir el archivo de imagen estándar
print("Sube tu segunda imagen médica (ej: histologia.png o endoscopia.jpg)")
uploaded_fourier = files.upload()
path_fourier_2 = list(uploaded_fourier.keys())[0]

# Leer en escala de grises
segunda_imagen = cv2.imread(path_fourier_2, cv2.IMREAD_GRAYSCALE)

# Procesar y mostrar
mostrar_galeria_fourier(segunda_imagen, "Imagen 2: Segunda Captura Médica")